# 第二轮困难正样本消融与蒸馏

固定原 Pilot 的 470/75/50 互斥划分；先训练三个监督模型，再提取 Teacher 并训练三个 KD 变体。模型和阈值仅用验证集选择，最终只对入选 Baseline 与 KD 各评估一次固定测试集。固定测试集已在上一轮被查看，因此不再视为盲测。

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('未分配Kaggle GPU')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = '3ca5c1258fc6a7bb163698bac7a80592f9c8fd97'
subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', fixed_commit], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit不一致: {actual_commit}')
print('GitHub commit:', actual_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(repo_dir / 'requirements.txt')], check=True)
config_path = repo_dir / 'configs/kaggle_pv_pilot_distill.yaml'
output_dir = Path('/kaggle/working/round2_outputs')

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(repo_dir / 'scripts' / name), *map(str, arguments)], cwd=repo_dir, check=True)

def run_student(variant, run_name, extra=(), evaluate_only=False, thresholds=None, test=False):
    command = ['--config', config_path, '--output-dir', output_dir, '--variant', variant, '--run-name', run_name]
    if evaluate_only:
        command.append('--evaluate-only')
        command.extend(['--thresholds', thresholds])
        if test:
            command.append('--test')
    else:
        command.extend(['--epochs', '12', '--defer-test', *extra])
    run_script('10_train_pilot_student.py', *command)


In [ ]:
run_script('08_prepare_pilot_manifest.py', '--config', config_path, '--output-dir', output_dir)
manifest = json.loads((output_dir / 'reports/pilot_manifest.json').read_text(encoding='utf-8'))
keys = [row['cache_key'] for rows in manifest['splits'].values() for row in rows]
if len(keys) != 595 or len(set(keys)) != 595:
    raise RuntimeError('划分数量或唯一性不符')
print(json.dumps(manifest['counts'], ensure_ascii=False, indent=2))


In [ ]:
baseline_runs = {
    'reference': (),
    'hard_weight': ('--hard-positive-weight', '2.5'),
    'positive_weight': ('--positive-pixel-weight', '2.0'),
}
for run_name, extra in baseline_runs.items():
    run_student('baseline', run_name, extra)
    run_student('baseline', run_name, evaluate_only=True, thresholds='0.3,0.4,0.5,0.6')
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'baseline')
selection_path = output_dir / 'reports/round2_selection.json'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
baseline = selection['baseline']
print('Selected baseline:', baseline['run_name'], 'threshold:', baseline['threshold'])


In [ ]:
run_script('09_extract_pilot_teacher.py', '--config', config_path, '--output-dir', output_dir)


In [ ]:
baseline_parameters = baseline['parameters']
supervision = ['--positive-pixel-weight', str(baseline_parameters['positive_pixel_weight'])]
if baseline_parameters['hard_positive_weight'] is not None:
    supervision.extend(['--hard-positive-weight', str(baseline_parameters['hard_positive_weight'])])
kd_runs = {
    'semantic_only': ('--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.0'),
    'feature_only': ('--semantic-kd-weight', '0.0', '--feature-kd-weight', '0.05'),
    'combined_low': ('--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.05'),
}
for run_name, kd_parameters in kd_runs.items():
    run_student('kd', run_name, (*supervision, *kd_parameters, '--feature-agreement-only'))
    run_student('kd', run_name, evaluate_only=True, thresholds='0.3,0.4,0.5,0.6')
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'kd')
selection = json.loads(selection_path.read_text(encoding='utf-8'))
print('Selected KD:', selection['kd']['run_name'], 'validation guard:', selection['kd_validation_guard_passed'])


In [ ]:
for variant, selected in (('baseline', selection['baseline']), ('kd', selection['kd'])):
    run_student(variant, selected['run_name'], evaluate_only=True, thresholds=str(selected['threshold']), test=True)
run_script('12_select_round2.py', '--output-dir', output_dir, '--stage', 'finalize')
selection = json.loads(selection_path.read_text(encoding='utf-8'))
print('Accepted:', selection['test_comparison']['accepted'])
print('Test deltas:', json.dumps(selection['test_comparison']['delta'], ensure_ascii=False, indent=2))
